# 01 - Exploring K8s Audit Logs for ML Security

This notebook explores Kubernetes audit log patterns relevant to ML infrastructure security.
We examine what normal vs malicious K8s API activity looks like in ML training clusters.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path

## Load Benchmark Dataset

Load the synthetic benchmark trajectories to analyze K8s audit patterns.

In [ ]:
with open('../benchmark/data/mlshield_benchmark_v1.json') as f:
    dataset = json.load(f)

print(f'Total trajectories: {len(dataset)}')

# Count by label
label_counts = Counter(t['label'] for t in dataset)
for label, count in sorted(label_counts.items()):
    print(f'  {label}: {count}')

## Flatten Events into DataFrame

In [ ]:
rows = []
for traj in dataset:
    for event in traj['events']:
        rows.append({
            'job_id': traj['job_id'],
            'label': traj['label'],
            'step': event.get('step', 0),
            'action': event.get('action', ''),
            'resource': event.get('resource', ''),
            'is_malicious': event.get('is_malicious', False),
            'violation_type': event.get('violation_type'),
        })

df = pd.DataFrame(rows)
print(f'Total events: {len(df):,}')
print(f'Malicious events: {df["is_malicious"].sum():,}')
df.head(10)

## K8s Action Distribution

Compare action types across benign vs malicious events.

In [ ]:
# Action distribution: benign vs malicious
benign_actions = df[~df['is_malicious']]['action'].value_counts()
malicious_actions = df[df['is_malicious']]['action'].value_counts()

print('=== Benign Event Actions ===')
print(benign_actions.head(10))
print()
print('=== Malicious Event Actions ===')
print(malicious_actions.head(10))

## Resource Access Patterns

Examine which resources are accessed in attack vs normal scenarios.

In [ ]:
# Resources accessed by malicious events
mal_resources = df[df['is_malicious']]['resource'].value_counts()
print('=== Resources Targeted in Attacks ===')
print(mal_resources.head(15))

print()
print('=== Violation Types ===')
print(df[df['is_malicious']]['violation_type'].value_counts())

## Attack Timeline Analysis

When do malicious events appear within trajectories?

In [ ]:
# At which step do attacks begin?
attack_trajs = df[df['label'] != 'benign']
first_malicious = attack_trajs[attack_trajs['is_malicious']].groupby('job_id')['step'].min()

print(f'Attack start step statistics:')
print(f'  Mean:   {first_malicious.mean():.1f}')
print(f'  Median: {first_malicious.median():.1f}')
print(f'  Min:    {first_malicious.min()}')
print(f'  Max:    {first_malicious.max()}')

print(f'\nAttack start step distribution:')
bins = pd.cut(first_malicious, bins=[0, 10, 20, 30, 40, 50])
print(bins.value_counts().sort_index())

## Weight Exfiltration Attack Phases

Examine the 4-phase pattern of weight exfiltration attacks.

In [ ]:
# Look at a single weight exfiltration trajectory
exfil_traj = next(t for t in dataset if t['label'] == 'weight_exfiltration')
print(f'Job: {exfil_traj["job_id"]}')
print(f'Attack starts at step: {exfil_traj.get("attack_start_step", "?")}')
print()

for event in exfil_traj['events']:
    if event.get('is_malicious'):
        print(f'  Step {event["step"]:3d} | {event["action"]:25s} | {event["resource"]}')

## Layer 1 Rule Coverage

Test which malicious events would be caught by static rules.

In [ ]:
from mlshield.specs.spec_parser import SpecParser
from mlshield.specs.spec_validator import SpecValidator
from mlshield.detectors.layer1_rules import RuleEngine
from mlshield.ingestion.event_bus import TrajectoryEvent, EventSource
from datetime import datetime, timezone

validator = SpecValidator(spec_path='../configs/default_specs.yaml')
rule_engine = RuleEngine(validator)

caught = 0
missed = 0
total_mal = 0

for traj in dataset[:200]:
    for event_data in traj['events']:
        if not event_data.get('is_malicious'):
            continue
        total_mal += 1
        
        event = TrajectoryEvent(
            event_id=f'{traj["job_id"]}-{event_data["step"]}',
            timestamp=datetime.now(timezone.utc),
            source=EventSource.K8S_AUDIT,
            job_id=traj['job_id'],
            user='test-user',
            action=event_data['action'],
            resource=event_data['resource'],
            details=event_data.get('details', {}),
            trajectory_step=event_data['step'],
        )
        result = rule_engine.check(event)
        if result.is_violation:
            caught += 1
        else:
            missed += 1

print(f'Layer 1 Rule Coverage on malicious events:')
print(f'  Caught: {caught}/{total_mal} ({caught/max(total_mal,1)*100:.1f}%)')
print(f'  Missed: {missed}/{total_mal} ({missed/max(total_mal,1)*100:.1f}%)')

## Summary

Key findings from exploring K8s audit log patterns:

1. **Normal training** is dominated by `gpu_metrics_snapshot`, `k8s_get`, and `k8s_create` actions
2. **Malicious events** introduce distinct actions: `k8s_exec`, `network_egress`, `k8s_list` on sensitive resources
3. **Weight exfiltration** follows a 4-phase pattern: reconnaissance, staging, conversion, exfiltration
4. **Attacks begin** at various steps (10-30), requiring temporal awareness for early detection
5. **Layer 1 rules** catch a significant portion of malicious events, but not all -- multi-layer detection is essential